# 02. Methods

**Tier:** Core. **Why you'd use this:** A CSV-ish line has spaces, and you need the words as a list, then one cleaned string again. Methods do that work, but they leave the original string alone.

**After this lesson you can:** use `strip`, `split`, and `join` on text; explain why a method leaves the original string in place; choose `join` when many pieces become one string.
About 20 minutes. **Uses:** [Indexing](01-indexing-lesson.ipynb)


## Quick review

1. What does `"abc"[0]` give you? What about `"abc"[1:3]`?
2. Why doesn't `"abc"[1:10]` raise `IndexError`?

<details><summary>Answers</summary>

`a` and `bc`. A slice stops at the end; a bad index raises.
</details>


## The idea

A method is a function called on a value: `value.method()`. `strip` drops space from both ends. `split` breaks one string into pieces. `join` builds one string, and you call it on the separator: `", ".join(words)`.

<details><summary>Explain more</summary>

`"Hello".lower()` returns `"hello"` and leaves the original alone. `split()` treats a run of spaces as one break. `split(" ")` keeps the empty pieces between single spaces.
</details>


## See it work

Turn a `"Last, First"` record into a byline.


In [1]:
def display_name(raw):
    # 1. Split on the comma. Each side is its own string.
    last, first = raw.split(",")
    # 2. Drop the space the comma left behind. strip returns a new string.
    last = last.strip()
    first = first.strip()
    # 3. One space between the names, and not after the last name.
    return " ".join([first, last])


print(display_name("Lovelace, Ada"))
print(display_name(" Hopper , Grace "))

Ada Lovelace
Grace Hopper


## Check yourself

Answer from the example before you run.


1. Why is `join` called on the separator rather than on the list?
2. What does `"  a  b  ".split()` drop that `split(" ")` keeps?

<details><summary>Answers</summary>

The separator is the string doing the work. `split()` collapses a run of spaces. `split(" ")` keeps an empty string for each extra space.
</details>


In [2]:
raw = "  Ada "
spaced = "a  b"
print(repr(raw.strip()))
print(repr(raw))
print(spaced.split())
print(spaced.split(" "))
print(" ".join(["Grace", "Hopper"]))

'Ada'
'  Ada '
['a', 'b']
['a', '', 'b']
Grace Hopper


<details><summary>Check your answers</summary>

- `strip` returns `'Ada'`. `raw` is still `'  Ada '`.
- `split()` gives `['a', 'b']`. `split(" ")` keeps the empty piece between the spaces.
- `join` puts one space between the names and nothing after the last.
</details>


## Why it works this way

`strip` and `lower` allocate a new string and return it. The original's code points are fixed, so a method has nowhere to write. Drop the return value and you drop the only copy of the result. [Numbers and strings](../../internals/05-numbers/01-numbers-strings-lesson.ipynb) shows the same rule for `replace`.

Adding one piece at a time copies a longer prefix each time — about n²/2. `"".join` sums lengths, allocates once, and copies each piece once.

The timed line calls `add`, so the left-hand string is still in use and must be copied. CPython detail: a bare `s += piece` can sometimes resize when nothing else points at it. `join` does not depend on that. Chained tiny transforms pay that allocation cost in a hot loop.


In [3]:
name = "ada"
print(name.upper() is name)
print(name.upper())
print(name)

False
ADA
ada


In [4]:
import timeit


def add(left, right):
    return left + right


print(
    "copy 20000",
    timeit.timeit(
        "text = add(text, 'a')",
        setup="text = ''",
        number=20000,
        globals=globals(),
    ),
)
print(
    "copy 80000",
    timeit.timeit(
        "text = add(text, 'a')",
        setup="text = ''",
        number=80000,
        globals=globals(),
    ),
)
print(
    "join 80000",
    timeit.timeit(
        "''.join(parts)",
        setup="parts = ['a'] * 80000",
        number=1,
    ),
)

copy 20000 0.0047875829914119095
copy 80000 0.08306304199504666
join 80000 0.0002503330179024488


## When to use it

Use `join` for many pieces; `+` for two or three. Use `split()` on whitespace, and `split(separator)` when empty pieces matter. `strip` only touches the ends.


## Common mistake

Calling `strip` and `lower` and then returning the original name. Both calls returned new strings, and nothing stored them.


In [5]:
def tidy(raw):
    raw.strip()
    raw.lower()
    return raw


print(repr(tidy("  Ada ")))

'  Ada '


Chain the calls and return that result. Rebinding the parameter does not change any other name that still points at the original.


In [6]:
def tidy(raw):
    return raw.strip().lower()


print(repr(tidy("  Ada ")))

'ada'


`join` only inserts text between text. A number has to become text first. The last traceback line is `TypeError`, and the arrow marks the `join`.


In [7]:
def listed(items):
    return ",".join(items)


print(listed(["a", 1]))

TypeError: sequence item 1: expected str instance, int found

## Try it

Change `display_name` so both names are lowercase in the byline. Which call returns that new string, and what does the original `raw` parameter still refer to if you print it before the return?


## Recap

- `strip` and `lower` return a new string. The original stays.
- `join` is called on the separator and allocates once.
- Repeated `+` copies a growing prefix.

Next: [practice](02-methods-practice.ipynb).
